In [5]:
from dotenv import load_dotenv
# loads the environment variables from a .env file into the system's environment variables
# This contains the API keys and other sensitive information that should not be hardcoded into the codebase.
# The API keys are used to authenticate the agent with the language model provider and other services.

load_dotenv()

True

In [12]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()
#TavilyClient is a client for the Tavily API, which allows you to search the web for information. It is used in the web_search tool to perform web searches.

# Decorator to register the function as a tool that can be used by the agent. The tool takes a query string as input and returns a dictionary containing the search results from the Tavily API.
# tool decorator is used to register the function as a tool that can be used by the agent. The tool takes a query string as input and returns a dictionary containing the search results from the Tavily API.
# Tools are functions that can be called by the agent to perform specific tasks, such as searching the web, retrieving data from a database, or performing calculations. The agent can use these tools to augment its capabilities and provide more accurate and relevant responses to user queries.
@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""
# LangSmith traces the execution of the tool, allowing you to see how the tool was called and what it returned. This can be useful for debugging and understanding how the agent is using the tool.
# My LangSmith API key is stored in the .env file and is loaded into the system's environment variables using the load_dotenv() function. This allows the agent to authenticate with the LangSmith API and use its tracing capabilities.
# LangSmith Tracing for this is : https://smith.langchain.com/o/a19995d9-0fdf-402c-a3ac-be4a34b95eb1/projects/p/e6566967-fb06-4a35-b245-0021ef30badb
    return tavily_client.search(query)

In [13]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [14]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [16]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [17]:
print(uploader.value)

({'name': 'LeftOver.png', 'type': 'image/png', 'size': 35885, 'content': <memory at 0x12185f940>, 'last_modified': datetime.datetime(2026, 10, 3, 23, 7, 42, 699000, tzinfo=datetime.timezone.utc)},)


In [40]:
import base64
import subprocess
import tempfile
from pathlib import Path

uploaded_file = uploader.value[0]
content_mv = uploaded_file["content"]
img_bytes = bytes(content_mv)
image_mime_type = "image/png"

if img_bytes[4:12] == b"ftypavif":
    with tempfile.TemporaryDirectory() as temp_dir:
        source_path = Path(temp_dir) / "upload.avif"
        output_path = Path(temp_dir) / "upload.jpg"
        source_path.write_bytes(img_bytes)
        subprocess.run(
            ["sips", "-s", "format", "jpeg", str(source_path), "--out", str(output_path)],
            check=True,
            capture_output=True,
        )
        img_bytes = output_path.read_bytes()
    image_mime_type = "image/jpeg"

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

In [44]:
from uuid import uuid4

image_config = {"configurable": {"thread_id": str(uuid4())}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "I have some leftover as shown in the picture. What can I make?"},
    {"type": "image", "base64": img_b64, "mime_type": image_mime_type}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config=image_config
)

print(response['messages'][-1].content)

Nice! It looks like you’ve got leftover spaghetti with sauce, some bread crusts/slices, and a bunch of scallions. Here are several tasty ideas that use those items (plus a few pantry staples you might already have). If you want, I can pull full recipe cards for any option.

1) Spaghetti frittata (pasta omelette)
- Why: uses your spaghetti with sauce, plus eggs and cheese if you have them.
- How: whisk 4 eggs with a bit of salt and pepper and some grated cheese (parmesan, mozzarella, or any you have). Mix in cooled leftover spaghetti. Heat a little olive oil in an ovenproof skillet, pour in the pasta-egg mixture, and cook until set on the bottom. Finish under the broiler or in a hot oven until the top is set and lightly golden.
- Quick version: mix spaghetti with beaten eggs and cheese in a bowl, then fry in a skillet and flip/finish until firm.

2) Fried leftover spaghetti
- Why: fast, crispy, and delicious with what you’ve got.
- How: melt a bit of butter in a skillet, add the leftove

In [35]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='c31f6ccc-5bca-43a8-a53e-2a9a2f6aa136'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 567, 'prompt_tokens': 199, 'total_tokens': 766, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EV33WhGNRcI0z0ynEJgjakecVsZiX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10402-4ae0-7ef2-b5e3-79e8d891af08-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'leftover chicken and rice recipes'}, 'id': 'call_oQlCFCypHj6bJacQxoWh6R6K', 't